# 00 · The data and the task: protein secondary structure

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LekshmiSR/proteins-d1-hands-on/blob/main/notebooks/00_data_and_task.ipynb)

**Goal of this notebook:** understand exactly what goes *into* our deep learning models and what must come *out*, before we build any model.

You will learn:
1. What protein secondary structure is and how it is labelled (DSSP).
2. Where the data comes from (the **PS4** dataset and the **CB513** test set).
3. How a protein sequence is turned into numbers (tokens, one-hot, embeddings).
4. Why we need **padding** and **masks** when proteins have different lengths.
5. How we will score a model (Q3 accuracy, F1, confusion matrix), and two simple baselines every model must beat.

No model is trained here. This notebook takes about 2 minutes to run and does not need a GPU.

## The biology in one minute

A protein is a chain of amino acids (20 kinds, written as letters: `A`, `C`, `D`, ... `Y`).
The chain folds into a 3D shape. Locally, stretches of the chain adopt a few repeating patterns, called **secondary structure**:

| Symbol | Name | What it looks like |
|---|---|---|
| **H** | α-helix | a spiral, held by hydrogen bonds every 4 residues |
| **E** | β-strand | an extended stretch that pairs side by side with other strands to form sheets |
| **C** | coil | everything else (loops, turns, bends) |

The program **DSSP** reads a solved 3D structure and assigns one of **8 states** to every residue (H, G, I, E, B, T, S, C, plus P for polyproline in PS4).
These are usually merged into the **3 states** above:

* H, G, I → **H** (helix)
* E, B → **E** (strand)
* everything else → **C** (coil)

**Our classification problem:** given only the amino-acid sequence, predict H / E / C for **every residue**.
This is called *per-residue* (or *token*) classification: one protein of length *L* gives *L* predictions.

## 0. Setup

Run the next cell first. It works in two places:

* **Google Colab**: it downloads this repository (code + helpers) into the Colab machine.
* **Your own computer** (Jupyter / VS Code): it just moves to the repository folder.

> **Colab tip:** use a GPU. Menu **Runtime → Change runtime type → T4 GPU → Save**. Do this *before* running cells.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/LekshmiSR/proteins-d1-hands-on.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if "google.colab" in sys.modules:                       # running in Colab
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.exists(REPO_DIR):
            subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":      # running locally from notebooks/
    os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from collections import Counter

from protein_dl import data

## 1. Download the data

We use two public datasets (about 10 MB in total, downloaded from the original PS4 GitHub repository):

* **PS4** ([Peracha et al., 2023](https://github.com/omarperacha/ps4-dataset), CC0 licence): 18,731 protein chains from the Protein Data Bank, filtered so that no two are too similar. It comes with an official train / validation split.
* **CB513**: a classic benchmark of 513 proteins (511 in this version) used in many papers. We use it **only as the final test set**. PS4 was built so that it does not overlap with CB513.

Why three sets?
* **train**: the model learns from these.
* **validation**: we watch these while training to choose settings and the best epoch.
* **test (CB513)**: touched only at the very end, to get an honest score.

In [ ]:
data.download_data("data")
train_df, valid_df, test_df = data.load_raw_tables("data")

print("train proteins:", len(train_df))
print("valid proteins:", len(valid_df))
print("test  proteins:", len(test_df), "(CB513)")
train_df.head()

Each row is one protein chain:
* `id`: PDB code + chain letter (e.g. `4trtA`)
* `sequence`: the amino acids, one letter each
* `ss8`: the DSSP 8-state label, **one letter per residue** (so it has the same length as the sequence)
* `mask`: `1` = this residue is scored. In PS4 all residues are scored. In CB513 only residues that belong to the original benchmark are scored.

Let us look at one protein, residue by residue.

In [ ]:
row = train_df.iloc[0]
print("id      :", row["id"])
print("length  :", len(row["sequence"]))
print("sequence:", row["sequence"][:70])
print("ss8     :", row["ss8"][:70])

## 2. Explore: protein lengths and class balance

In [ ]:
lengths = train_df["sequence"].str.len()
print(lengths.describe().round(1))

plt.figure(figsize=(7, 3))
plt.hist(lengths, bins=80)
plt.axvline(512, color="red", ls="--", label="512 (crop length used for training)")
plt.xlabel("protein length (residues)"); plt.ylabel("number of proteins"); plt.legend()
plt.title("Training set: protein lengths")
plt.show()

Most proteins are shorter than 512 residues. During training we randomly **crop** the few longer ones to 512 residues. This keeps GPU memory small (important for a 4 GB GPU) and changes very little.

Now the labels. First the 8 DSSP states, then after merging to 3 states:

In [ ]:
q8_counts = Counter("".join(train_df["ss8"]))
q3_counts = Counter(data.Q8_TO_Q3[c] for c in "".join(train_df["ss8"]))
total = sum(q8_counts.values())

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
k8 = sorted(q8_counts, key=q8_counts.get, reverse=True)
axes[0].bar(k8, [q8_counts[k] / total for k in k8])
axes[0].set_title("8 DSSP states"); axes[0].set_ylabel("fraction of residues")
axes[1].bar(data.Q3_NAMES, [q3_counts[c] / total for c in data.Q3_CLASSES], color=["grey", "tab:red", "tab:blue"])
axes[1].set_title("3 states (our task)")
plt.show()

for c, name in zip(data.Q3_CLASSES, data.Q3_NAMES):
    print(f"{name:6s} ({c}): {q3_counts[c] / total:.1%}")

The classes are **not balanced**: strands (E) are the rarest. Keep this in mind: a model can get a decent accuracy while being poor at strands, which is why we will also look at per-class scores.

## 3. From letters to numbers

Neural networks only work with numbers. We turn every amino acid into an integer **token**:

* `0` is reserved for **padding** (explained below)
* `1` ... `20` are the 20 standard amino acids
* `21` means "unknown / non-standard" (e.g. `X`)

and every label into a class index: `C → 0`, `H → 1`, `E → 2`.

In [ ]:
print("vocabulary:", data.AA_TO_IDX)
print("vocab size (incl. PAD and UNK):", data.VOCAB_SIZE)

seq = row["sequence"][:12]
ss8 = row["ss8"][:12]
tokens = data.encode_sequence(seq)
labels = data.encode_q8_as_q3(ss8)

print()
print("sequence :", list(seq))
print("tokens   :", tokens)
print("ss8      :", list(ss8))
print("ss3      :", [data.Q8_TO_Q3[c] for c in ss8])
print("labels   :", labels)

### One-hot vectors and embeddings

An integer like `13` has no meaning as a *size* (methionine is not "bigger" than lysine). So models never use the token number directly. Two common choices:

1. **One-hot**: token `k` becomes a vector of zeros with a single `1` at position `k`. Length = vocabulary size (22).
2. **Embedding** (`nn.Embedding`): a learnable table with one row per token. Token `k` simply picks row `k`. The model learns useful numbers for each amino acid during training (e.g. it may learn that hydrophobic residues are similar).

An embedding is mathematically the same as *one-hot × weight matrix*, just faster. All our models start with an embedding layer. Let us check the shapes.

In [ ]:
x = torch.tensor(tokens)                       # shape [L]
print("tokens            :", x.shape)

one_hot = torch.nn.functional.one_hot(x, num_classes=data.VOCAB_SIZE).float()
print("one-hot           :", one_hot.shape, "  <- [L, vocab]")

emb = torch.nn.Embedding(num_embeddings=data.VOCAB_SIZE, embedding_dim=8, padding_idx=data.PAD_IDX)
e = emb(x)
print("embedding         :", e.shape, "  <- [L, embedding_dim]")

# The same thing via one-hot @ weight matrix:
e2 = one_hot @ emb.weight
print("one_hot @ weight == embedding ?", torch.allclose(e, e2))

## 4. Batches, padding and masks

GPUs are fast when they process many proteins at once, as one rectangular tensor (a **batch**).
But proteins have different lengths. Solution:

* **Padding**: append `0` tokens to the shorter proteins so all have the length of the longest one in the batch.
* **Ignored labels**: padded positions get the label `-100`. PyTorch's loss function skips any position with label `-100`, and so do our metrics.
* **Lengths**: we also keep each protein's true length, because some models (the LSTM) need it.

Here is a tiny batch of 3 proteins, built by our `ProteinSSDataset` and `collate_batch` helpers:

In [ ]:
tiny = data.ProteinSSDataset(train_df.iloc[:3])
items = [tiny[i] for i in range(3)]
for i, (x, y) in enumerate(items):
    print(f"protein {i}: tokens {tuple(x.shape)}, labels {tuple(y.shape)}")

tokens, labels, lengths = data.collate_batch(items)
print()
print("tokens  :", tokens.shape, "  <- [batch, longest length]")
print("labels  :", labels.shape)
print("lengths :", lengths)
print()
i = int(lengths.argmin())
print(f"end of the shortest protein (#{i}), tokens :", tokens[i, lengths[i] - 3: lengths[i] + 5].tolist())
print(f"end of the shortest protein (#{i}), labels :", labels[i, lengths[i] - 3: lengths[i] + 5].tolist())

You can see the real residues followed by padding: tokens `0` and labels `-100`.

### DataLoaders

`make_dataloaders` does all of the above for the whole dataset and gives three PyTorch `DataLoader`s. Each time you loop over a loader you get batches `(tokens, labels, lengths)`.
* The **training** loader shuffles proteins and crops long ones to 512.
* The **validation/test** loaders keep full proteins and group similar lengths together (less padding).

In [ ]:
train_dl, valid_dl, test_dl = data.make_dataloaders("data", batch_size=32, max_len=512)

tokens, labels, lengths = next(iter(train_dl))
print("batches per epoch:", len(train_dl))
print("tokens :", tokens.shape, tokens.dtype)
print("labels :", labels.shape, labels.dtype)
print("lengths:", lengths.shape, lengths[:8].tolist(), "...")
print("real residues in this batch:", int((labels != -100).sum()), "of", labels.numel(), "positions")

**Every model in this repository has the same contract:**

```
input : tokens  [B, L]      (B = batch size, L = longest protein in the batch)
        lengths [B]
output: logits  [B, L, 3]   one score per class (C, H, E) for every residue
```

A *logit* is an unnormalised score. `softmax` turns the 3 scores of a residue into probabilities, and the predicted class is the one with the highest score (`argmax`).

## 5. How we score models

* **Q3 accuracy** = fraction of residues whose predicted class is correct. This is the standard number reported in papers.
* **Per-class precision / recall / F1**: how good the model is for H, E and C separately (important because E is rare).
* **Macro F1**: the average of the three F1 scores (treats each class as equally important).
* **Confusion matrix**: rows = true class, columns = predicted class. It shows *which* mistakes are made.

### Two baselines

Before training anything, let us see what very simple rules achieve. A deep model that cannot beat these has learned nothing useful.

1. **Majority class**: always predict the most common class.
2. **Amino-acid propensity**: for each amino acid, predict the class it is most often found in (e.g. alanine is often in helices). This looks at **one residue only**, with no context.

In [ ]:
from protein_dl.train import compute_metrics

train_ds = train_dl.dataset
test_ds = test_dl.dataset

# Count, for every amino-acid token, how often it is C, H or E in the training set
counts = np.zeros((data.VOCAB_SIZE, data.NUM_CLASSES))
for x, y in zip(train_ds.tokens, train_ds.labels):
    for t, lab in zip(x, y):
        if lab != data.IGNORE_INDEX:
            counts[t, lab] += 1

majority = int(counts.sum(0).argmax())
propensity = counts.argmax(1)               # best class for each amino acid

def run_rule(rule):
    trues, preds = [], []
    for x, y in zip(test_ds.tokens, test_ds.labels):
        x, y = np.array(x), np.array(y)
        keep = y != data.IGNORE_INDEX
        trues.append(y[keep]); preds.append(rule(x)[keep])
    return compute_metrics(trues, preds)

base_major = run_rule(lambda x: np.full(len(x), majority))
base_prop = run_rule(lambda x: propensity[x])

print(f"Majority class ({data.Q3_NAMES[majority]}):  CB513 Q3 = {base_major['q3_accuracy']:.3f}")
print(f"Amino-acid propensity:      CB513 Q3 = {base_prop['q3_accuracy']:.3f}, macro F1 = {base_prop['macro_f1']:.3f}")
print()
print("Propensity rule, class chosen for each amino acid:")
print({aa: data.Q3_CLASSES[propensity[i]] for aa, i in data.AA_TO_IDX.items()})

In [ ]:
from protein_dl import plots
plots.plot_confusion(base_prop["confusion_matrix"], "Propensity baseline on CB513")

The propensity baseline gets just under 50%, barely better than always guessing coil, and the confusion matrix shows it confuses the classes a lot. Knowing **one** residue is not enough: whether a residue is in a helix depends on its **neighbours**. The architectures in the next notebooks are different ways of letting the model look at that context:

| Notebook | Architecture | How it sees the neighbours |
|---|---|---|
| 01 | **MLP** with a sliding window | a fixed window of ±7 residues, flattened into one vector |
| 02 | **1D CNN** | stacked convolution filters that slide along the chain |
| 03 | **Bidirectional LSTM** | reads the chain left→right and right→left, carrying a memory |
| 04 | **Transformer** | self-attention: every residue can look at every other residue |
| 05 | **Comparison** | all models side by side |

For reference: models that use only the single sequence (like ours) reach about 65–75% Q3 on CB513. The best published methods reach about 85–88% by adding evolutionary information or large pretrained protein language models.

## Exercises

1. Change `train_df.iloc[0]` to another index and look at a different protein. Can you spot helices (long runs of H) and strands?
2. What fraction of the CB513 residues is masked out? (Hint: count `-100` labels in `test_ds.labels`.)
3. Compute the propensity baseline on the **validation** set instead of CB513. Is it similar?